# TherMAM-NeRF → PINN Bioheat Pipeline
**Full pipeline:** NeRF volumetric extraction → Gaussian smoothing → Marching cubes → Trimesh cleanup → PINN inverse bioheat solver → FEA forward verification

- Geometry source: `thermamnerf_best.pth` (SiameseEncoder + ThermamNeRFMLP)
- Temperature: denormalized to absolute °C using frontal-view calibration
- PINN: Pennes bioheat inverse solver (learnable x₀, y₀, z₀, r, Q_max)
- FEA: FEniCSx forward verification, target residual < 1.5°C

## 0 · Imports

In [ ]:
import os, sys, math, time, random, struct
import numpy as np
import tifffile
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from PIL import Image
from scipy.ndimage import gaussian_filter, map_coordinates
from scipy.spatial import cKDTree
from skimage.measure import marching_cubes
import plotly.graph_objects as go
import plotly.io as pio

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

pio.renderers.default = 'notebook'  # change to 'browser' for SSH/headless

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1 · Path configuration

In [ ]:
# ── Adjust to your server layout ─────────────────────────────────────────────
REPO_ROOT   = Path('.').resolve().parent
TIFF_DIR    = str(REPO_ROOT / 'data' / 'organized_by_patient')
UNET_DIR    = str(REPO_ROOT / 'data' / 'organized_by_patient_unet')
NERF_CKPT   = str(REPO_ROOT / 'thermamnerf_outputs2.9' / 'thermamnerf_best.pth')
RESULTS_DIR = Path(REPO_ROOT / 'UNET_Segmentation' / 'PINNpdeSolver' / 'results')
STL_DIR     = Path(REPO_ROOT / 'UNET_Segmentation' / 'PINNpdeSolver' / 'exported_stls')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
STL_DIR.mkdir(parents=True, exist_ok=True)

print(f'TIFF dir exists  : {Path(TIFF_DIR).exists()}')
print(f'UNET dir exists  : {Path(UNET_DIR).exists()}')
print(f'NeRF ckpt exists : {Path(NERF_CKPT).exists()}')

## 2 · TherMAM-NeRF config & model definitions

In [ ]:
CFG = {
    'img_size'        : 128,
    'n_views'         : 5,
    'view_angles_deg' : [-90, -45, 0, 45, 90],
    'view_names'      : ['RL', 'RO', 'F', 'LO', 'LL'],
    'feat_channels'   : 32,
    'pos_enc_L'       : 8,
    'mlp_hidden'      : 256,
    'mlp_layers'      : 4,
    'n_samples'       : 256,
    'near'            : -1.0,
    'far'             :  1.0,
    'density_scale'   : 10.0,
    'mc_threshold'    : 0.3,
    'mc_resolution'   : 128,
    'breast_radius_mm': 70.0,   # physical scale (Jahani et al., 2023)
}

# ── SiameseEncoder (verbatim from thermamnerf_v2_9.py) ──
class SiameseEncoder(nn.Module):
    def __init__(self, out_channels=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2,  16, 3, padding=1), nn.GroupNorm(4, 16),  nn.ReLU(inplace=False),
            nn.Conv2d(16, 32, 3, padding=1), nn.GroupNorm(8, 32),  nn.ReLU(inplace=False),
            nn.Conv2d(32, 32, 3, padding=1), nn.GroupNorm(8, 32),  nn.ReLU(inplace=False),
            nn.Conv2d(32, out_channels, 1),
        )
    def forward(self, tiff_norm, mask):
        x = torch.stack([tiff_norm, mask], dim=1)
        return self.net(x) * mask.unsqueeze(1)

# ── ThermamNeRFMLP (verbatim from thermamnerf_v2_9.py) ──
class ThermamNeRFMLP(nn.Module):
    def __init__(self, pos_enc_dim, feat_dim, hidden=256, n_layers=6):
        super().__init__()
        in_dim = pos_enc_dim + feat_dim
        self.layers  = nn.ModuleList()
        self.skip_at = n_layers // 2 - 1
        prev = in_dim
        for i in range(n_layers - 1):
            self.layers.append(
                nn.Linear(prev + in_dim if i == self.skip_at else prev, hidden)
            )
            prev = hidden
        self.sigma_head = nn.Linear(hidden, 1)
        self.temp_head  = nn.Linear(hidden, 1)
    def forward(self, pe, feat):
        x0 = torch.cat([pe, feat], dim=-1)
        h  = x0
        for i, layer in enumerate(self.layers):
            if i == self.skip_at: h = torch.cat([h, x0], dim=-1)
            h = F.relu(layer(h), inplace=False)
        return F.softplus(self.sigma_head(h)), torch.sigmoid(self.temp_head(h))

def positional_encoding(x, L, alpha=None):
    freqs = 2.0 ** torch.arange(L, dtype=torch.float32, device=x.device)
    x_freq = x.unsqueeze(-1) * freqs * math.pi
    sin_p, cos_p = torch.sin(x_freq), torch.cos(x_freq)
    if alpha is not None:
        k = torch.arange(L, dtype=torch.float32, device=x.device)
        w = 0.5 * (1 - torch.cos(math.pi * torch.clamp(alpha - k, 0., 1.)))
        sin_p = sin_p * w.view(1, 1, 1, L)
        cos_p = cos_p * w.view(1, 1, 1, L)
    return torch.cat([sin_p, cos_p], dim=-1).flatten(-2)

def project_and_sample(pts_3d, feat_maps, view_angles_rad):
    B, N, _ = pts_3d.shape
    V = feat_maps.shape[1]
    per_view = []
    for v in range(V):
        theta = view_angles_rad[v]
        xr = torch.cos(theta) * pts_3d[...,0] + torch.sin(theta) * pts_3d[...,2]
        yr = pts_3d[..., 1]
        grid = torch.stack([xr, yr], dim=-1).unsqueeze(1)
        sampled = F.grid_sample(feat_maps[:,v], grid, mode='bilinear',
                                padding_mode='zeros', align_corners=True)
        per_view.append(sampled.squeeze(2).permute(0,2,1))
    stacked = torch.stack(per_view, dim=2)
    return torch.cat([stacked.mean(dim=2), stacked.var(dim=2)], dim=-1)

print('Model classes defined.')

## 3 · Data utilities & dataset

In [ ]:
def load_tiff_celsius(path, target_size):
    arr = tifffile.imread(str(path)).astype(np.float32)
    if arr.ndim == 3: arr = arr[..., 0]
    img = Image.fromarray(arr, mode='F').resize((target_size, target_size), Image.BILINEAR)
    return np.array(img, dtype=np.float32)

def load_mask(path, target_size):
    img = Image.open(str(path)).convert('L').resize((target_size, target_size), Image.NEAREST)
    return (np.array(img, dtype=np.float32) / 255.0 > 0.5).astype(np.float32)

def normalize_thermal(arr):
    tmin, tmax = arr.min(), arr.max()
    return (arr - tmin) / (tmax - tmin + 1e-6), tmin, tmax

def get_view_key(filename):
    n = filename.lower()
    if 'right later' in n: return 'RL'
    if 'right obli'  in n: return 'RO'
    if 'frontal' in n or 'anterior' in n: return 'F'
    if 'left obliq'  in n: return 'LO'
    if 'left later'  in n: return 'LL'
    return None

def discover_patients(tiff_base, unet_base):
    tb, ub = Path(tiff_base), Path(unet_base)
    pd_ = {}
    for tp in tb.rglob('*.tiff'):
        parts = tp.relative_to(tb).parts
        if len(parts) < 2: continue
        pid, lab, fn = parts[0], parts[1], parts[-1]
        vk = get_view_key(fn)
        if not vk: continue
        key = (pid, lab)
        if key not in pd_: pd_[key] = {'tiffs': {}, 'masks': {}}
        pd_[key]['tiffs'][vk] = tp
    for mp in ub.rglob('*.png'):
        parts = mp.relative_to(ub).parts
        if len(parts) < 2: continue
        pid, lab, fn = parts[0], parts[1], parts[-1]
        vk = get_view_key(fn)
        if not vk: continue
        key = (pid, lab)
        if key in pd_: pd_[key]['masks'][vk] = mp
    patients = []
    for (pid, lab), d in pd_.items():
        if len(d['tiffs']) == 5 and len(d['masks']) == 5:
            patients.append({'id': pid, 'label': lab,
                             'tiffs': d['tiffs'], 'masks': d['masks']})
    patients.sort(key=lambda p: p['id'])
    print(f'Found {len(patients)} complete patients.')
    return patients

class BreastThermDataset(Dataset):
    def __init__(self, patient_list, cfg):
        self.patients   = patient_list
        self.cfg        = cfg
        self.S          = cfg['img_size']
        self.view_names = cfg['view_names']

    def __len__(self): return len(self.patients)

    def __getitem__(self, idx):
        p = self.patients[idx]
        tiffs_norm, tiffs_abs, masks, tmins, tmaxs = [], [], [], [], []
        for v in self.view_names:
            raw         = load_tiff_celsius(str(p['tiffs'][v]), self.S)
            normd, tmin, tmax = normalize_thermal(raw)
            mask        = load_mask(str(p['masks'][v]), self.S)
            tiffs_norm.append(normd)
            tiffs_abs.append(raw)
            masks.append(mask)
            tmins.append(tmin)
            tmaxs.append(tmax)
        return {
            'patient_id' : p['id'],
            'label'      : p.get('label', 'unknown'),
            'tiffs_norm' : torch.tensor(np.stack(tiffs_norm), dtype=torch.float32),
            'tiffs_abs'  : torch.tensor(np.stack(tiffs_abs),  dtype=torch.float32),
            'masks'      : torch.tensor(np.stack(masks),      dtype=torch.float32),
            'tmin'       : torch.tensor(tmins, dtype=torch.float32),
            'tmax'       : torch.tensor(tmaxs, dtype=torch.float32),
        }

patients = discover_patients(TIFF_DIR, UNET_DIR)
dataset  = BreastThermDataset(patients, CFG)
print(f'Dataset ready: {len(dataset)} patients')

## 4 · Load NeRF checkpoint

In [ ]:
L        = CFG['pos_enc_L']
pos_dim  = 3 * 2 * L
feat_dim = CFG['feat_channels'] * 2

encoder = SiameseEncoder(out_channels=CFG['feat_channels']).to(device).eval()
mlp     = ThermamNeRFMLP(pos_enc_dim=pos_dim, feat_dim=feat_dim,
                          hidden=CFG['mlp_hidden'],
                          n_layers=CFG['mlp_layers']).to(device).eval()

ckpt = torch.load(NERF_CKPT, map_location=device)
encoder.load_state_dict({k.replace('module.', ''): v
                         for k, v in ckpt['encoder'].items()})
mlp.load_state_dict(    {k.replace('module.', ''): v
                         for k, v in ckpt['mlp'].items()})

n_params = sum(p.numel() for p in list(encoder.parameters()) +
               list(mlp.parameters()))
print(f'NeRF checkpoint loaded.  Total params: {n_params:,}')

## 5 · NeRF volumetric extraction

In [ ]:
@torch.no_grad()
def extract_3d_volume(encoder, mlp, tiffs_norm, masks, cfg, device,
                      resolution=None, chunk=8192):
    """
    tiffs_norm, masks: unbatched [V, H, W] tensors on device.
    Returns sigma_grid, T_grid — each shape [R, R, R] numpy.
    """
    R = resolution or cfg['mc_resolution']
    linspace = torch.linspace(-1, 1, R, device=device)
    zz, yy, xx = torch.meshgrid(linspace, linspace, linspace, indexing='ij')
    pts = torch.stack([xx, yy, zz], dim=-1).reshape(1, -1, 3)

    view_angles_rad = torch.tensor(
        [math.radians(a) for a in cfg['view_angles_deg']], device=device)
    feat_maps = torch.stack([
        encoder(tiffs_norm[v:v+1], masks[v:v+1])
        for v in range(cfg['n_views'])
    ], dim=1)

    alpha_final = float(cfg['pos_enc_L'])
    sigma_all, T_all = [], []
    for i in range(0, pts.shape[1], chunk):
        p    = pts[:, i:i+chunk]
        pe   = positional_encoding(p, L=cfg['pos_enc_L'], alpha=alpha_final)
        feat = project_and_sample(p, feat_maps, view_angles_rad)
        sg, tp = mlp(pe, feat)
        sigma_all.append(sg.squeeze().cpu())
        T_all.append(tp.squeeze().cpu())

    sigma_grid = torch.cat(sigma_all).reshape(R, R, R).numpy()
    T_grid     = torch.cat(T_all).reshape(R, R, R).numpy()
    return sigma_grid, T_grid

print('extract_3d_volume ready.')

## 6 · Geometry preprocessing: smooth → mesh → trimesh cleanup → mm conversion

In [ ]:
def sample_interior_points(verts_mm, n_points=5000):
    """Uniform random samples inside the bounding box of the mesh."""
    bbox_min = verts_mm.min(axis=0)
    bbox_max = verts_mm.max(axis=0)
    centroid = verts_mm.mean(axis=0)
    extents  = (bbox_max - bbox_min) / 2
    pts = np.random.uniform(bbox_min, bbox_max, size=(n_points * 3, 3))
    rel = np.abs(pts - centroid) / (extents + 1e-8)
    inside = np.all(rel < 1.0, axis=1)
    selected = pts[inside][:n_points]
    # Pad if not enough points
    if len(selected) < n_points:
        extra = np.random.uniform(bbox_min, bbox_max,
                                   size=(n_points - len(selected), 3))
        selected = np.vstack([selected, extra])
    return selected.astype(np.float32)

def process_patient_geometry_from_nerf(sigma_grid, T_grid, tmin, tmax, cfg,
                                        gaussian_sigma=1.0, n_interior=5000):
    """
    NeRF grids → geometry dict ready for train_pinn_single().

    Temperature calibration: uses frontal-view tmin/tmax as anchor
    (frontal view is most thermally representative — Bezerra et al., 2013).
    Gaussian smoothing is applied to sigma_grid ONLY; T_grid is untouched
    so thermal accuracy is preserved.
    """
    R  = cfg['mc_resolution']
    BR = cfg['breast_radius_mm']   # voxel [-1,1] → physical mm

    # ── Step 1: Smooth sigma only (geometry), leave T_grid raw ──
    sigma_smooth = gaussian_filter(sigma_grid, sigma=gaussian_sigma)

    # ── Step 2: Marching cubes in voxel space ──
    verts_vox, faces, normals, _ = marching_cubes(
        sigma_smooth, level=cfg['mc_threshold']
    )

    # ── Step 3: Convert voxel → normalised [-1,1] → physical mm ──
    verts_norm = verts_vox / (R - 1) * 2.0 - 1.0
    verts_mm   = verts_norm * BR

    # ── Step 4: Trimesh cleanup — largest component, watertight ──
    try:
        import trimesh
        mesh = trimesh.Trimesh(vertices=verts_mm, faces=faces, process=False)
        components = mesh.split(only_watertight=False)
        mesh = max(components, key=lambda m: len(m.faces))
        trimesh.smoothing.filter_laplacian(mesh, lamb=0.3, iterations=5)
        verts_mm = np.array(mesh.vertices, dtype=np.float32)
        faces    = np.array(mesh.faces)
        # Re-map T samples to cleaned vertex positions
        verts_norm_clean = verts_mm / BR
        verts_vox_clean  = (verts_norm_clean + 1.0) / 2.0 * (R - 1)
        print(f'  trimesh: {len(verts_mm)} verts, '
              f'watertight={mesh.is_watertight}')
    except ImportError:
        print('  trimesh not installed — skipping cleanup')
        verts_vox_clean = verts_vox

    # ── Step 5: Sample T at surface vertices from ORIGINAL T_grid ──
    # T_grid is [0,1] normalised; denormalize with frontal-view calibration
    temp_norm  = map_coordinates(T_grid, verts_vox_clean.T,
                                  order=1, mode='nearest').clip(0.0, 1.0)
    T_measured = (temp_norm * (tmax - tmin) + tmin).astype(np.float32)  # → °C

    # Uniform confidence (NeRF already fused all 5 views)
    confidence = np.ones(len(verts_mm), dtype=np.float32)

    # ── Step 6: Interior collocation points ──
    interior_pts = sample_interior_points(verts_mm, n_points=n_interior)

    bbox_min = verts_mm.min(axis=0)
    bbox_max = verts_mm.max(axis=0)

    return {
        'surface_pts' : verts_mm.astype(np.float32),
        'T_measured'  : T_measured,
        'confidence'  : confidence,
        'interior_pts': interior_pts,
        'bbox_min'    : bbox_min.astype(np.float32),
        'bbox_max'    : bbox_max.astype(np.float32),
        'bbox_extents': (bbox_max - bbox_min).astype(np.float32),
        'verts_raw'   : verts_mm.astype(np.float32),
        'faces'       : faces,
    }

# ── Quick test ──
sample   = dataset[0]
tiffs_n  = sample['tiffs_norm'].to(device)
masks_   = sample['masks'].to(device)
# Frontal view (index 2) as temperature anchor
tmin_ref = sample['tmin'][2].item()
tmax_ref = sample['tmax'][2].item()

print(f'Extracting test volume for {sample["patient_id"]} …')
sg, tg = extract_3d_volume(encoder, mlp, tiffs_n, masks_, CFG, device)
geo    = process_patient_geometry_from_nerf(sg, tg, tmin_ref, tmax_ref, CFG)
geo['patient_id'] = sample['patient_id']
geo['label']      = sample['label']

print(f'Surface pts : {geo["surface_pts"].shape}')
print(f'T_measured  : {geo["T_measured"].min():.1f} – '
      f'{geo["T_measured"].max():.1f} °C  (expect 28–38°C)')
print(f'Interior pts: {geo["interior_pts"].shape}')
print(f'BBox (mm)   : {geo["bbox_min"]} → {geo["bbox_max"]}')

## 7 · Quick 3D Plotly viewer (Option A)

In [ ]:
def plot_thermal_mesh(geo, colorscale='Inferno', opacity=0.85,
                      save_html=True):
    verts      = geo['surface_pts']
    faces      = geo['faces']
    temp_vals  = geo['T_measured']
    patient_id = geo['patient_id']

    fig = go.Figure(data=[go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=temp_vals,
        intensitymode='vertex',
        colorscale=colorscale,
        cmin=temp_vals.min(), cmax=temp_vals.max(),
        colorbar=dict(title='Temp (°C)', thickness=18),
        opacity=opacity,
        flatshading=False,
        lighting=dict(ambient=0.3, diffuse=0.8, specular=0.4,
                      roughness=0.5, fresnel=0.2),
        lightposition=dict(x=1, y=2, z=3),
        name=patient_id,
    )])
    fig.update_layout(
        title=dict(
            text=f'TherMAM-NeRF — {patient_id}<br>'
                 f'<sup>{len(verts):,} vertices | '
                 f'T={temp_vals.min():.1f}–{temp_vals.max():.1f}°C</sup>',
            font=dict(size=15)
        ),
        scene=dict(
            aspectmode='data',
            xaxis=dict(title='X (RL↔LL) mm'),
            yaxis=dict(title='Y (Inf↑Sup) mm'),
            zaxis=dict(title='Z (Depth) mm'),
            camera=dict(eye=dict(x=1.6, y=0.5, z=1.2)),
        ),
        margin=dict(l=0, r=0, b=0, t=60),
        width=920, height=700,
    )
    if save_html:
        html_path = RESULTS_DIR / f"{patient_id}_3d_thermal.html"
        fig.write_html(str(html_path), include_plotlyjs='cdn')
        print(f'Saved → {html_path}')
    return fig

fig = plot_thermal_mesh(geo)
fig.show()

## 8 · STL export (binary)

In [ ]:
def save_stl_binary(filepath, verts, faces):
    with open(filepath, 'wb') as f:
        f.write(b'\0' * 80)
        f.write(struct.pack('<I', len(faces)))
        for face in faces:
            tri = verts[face].astype(np.float32)
            v0, v1, v2 = tri
            normal = np.cross(v1-v0, v2-v0)
            norm   = np.linalg.norm(normal)
            normal = (normal/norm).astype(np.float32) if norm > 0 \
                     else np.zeros(3, dtype=np.float32)
            f.write(struct.pack('<3f', *normal))
            f.write(struct.pack('<3f', *v0))
            f.write(struct.pack('<3f', *v1))
            f.write(struct.pack('<3f', *v2))
            f.write(struct.pack('<H', 0))
    print(f'STL saved → {filepath}')

stl_path = STL_DIR / f"{geo['patient_id']}.stl"
save_stl_binary(str(stl_path), geo['surface_pts'], geo['faces'])

## 9 · PINN biophysical constants & model

In [ ]:
# ── Pennes bioheat constants (Table 2.2 — Bezerra et al. 2013; Das & Laxmi 2026) ──
K_TISSUE   = 0.48      # W/(m·K)
OMEGA_B    = 0.0005    # 1/s
C_BLOOD    = 3600.0    # J/(kg·K)
T_ARTERIAL = 37.0      # °C
Q_METAB    = 450.0     # W/m³

class BioheatPINN(nn.Module):
    """6-layer MLP with Tanh, maps (x,y,z)→T. Learnable tumour params."""
    def __init__(self, hidden=256, depth=6):
        super().__init__()
        layers = [nn.Linear(3, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)
        # Learnable tumour parameters (in normalised [-1,1] coords)
        self.x_t   = nn.Parameter(torch.tensor([0.0]))
        self.y_t   = nn.Parameter(torch.tensor([0.0]))
        self.z_t   = nn.Parameter(torch.tensor([0.0]))
        self.r_t   = nn.Parameter(torch.tensor([10.0]))   # mm
        self.Q_max = nn.Parameter(torch.tensor([5000.0])) # W/m³

    def forward(self, xyz):
        return self.net(xyz).squeeze(-1)

    def Q_tumor(self, xyz):
        d2 = ((xyz[:,0]-self.x_t)**2 +
              (xyz[:,1]-self.y_t)**2 +
              (xyz[:,2]-self.z_t)**2)
        return self.Q_max * torch.exp(-d2 / (self.r_t**2 + 1e-8))

def normalise_coords(pts, bbox_min, bbox_max):
    centre = (bbox_max + bbox_min) / 2
    extent = (bbox_max - bbox_min) / 2 + 1e-8
    return (pts - centre) / extent

def denormalise_coord(val, lo, hi):
    return val * (hi - lo) / 2 + (hi + lo) / 2

def compute_laplacian(T, xyz, extent_meters):
    """∇²T via autograd. xyz must have requires_grad=True."""
    dT  = torch.autograd.grad(T, xyz,
              grad_outputs=torch.ones_like(T), create_graph=True)[0]
    lap = sum(
        torch.autograd.grad(dT[:,i], xyz,
            grad_outputs=torch.ones_like(dT[:,i]),
            create_graph=True)[0][:,i] / (extent_meters[i]**2)
        for i in range(3)
    )
    return lap

def classify_quadrant(x_mm, y_mm):
    if x_mm > 0 and y_mm > 0: return 'Upper Outer'
    if x_mm < 0 and y_mm > 0: return 'Upper Inner'
    if x_mm > 0 and y_mm < 0: return 'Lower Outer'
    return 'Lower Inner'

print('PINN model & biophysical constants ready.')

## 10 · PINN training (Adam → L-BFGS, multi-start)

In [ ]:
def train_pinn_single(geo, device, n_starts=3,
                       adam_steps=3000, lbfgs_steps=100,
                       save_plot=True):
    surf_pts = torch.tensor(geo['surface_pts'], dtype=torch.float32, device=device)
    T_meas   = torch.tensor(geo['T_measured'],  dtype=torch.float32, device=device)
    conf     = torch.tensor(geo['confidence'],   dtype=torch.float32, device=device)
    int_pts  = torch.tensor(geo['interior_pts'], dtype=torch.float32, device=device)
    bbox_min = torch.tensor(geo['bbox_min'],     dtype=torch.float32, device=device)
    bbox_max = torch.tensor(geo['bbox_max'],     dtype=torch.float32, device=device)

    extent_mm     = (bbox_max - bbox_min) / 2 + 1e-8
    extent_meters = extent_mm * 1e-3

    surf_norm = normalise_coords(surf_pts, bbox_min, bbox_max)
    int_norm  = normalise_coords(int_pts,  bbox_min, bbox_max)

    best_loss, best_state, best_params, best_history = \
        float('inf'), None, None, None

    for start in range(n_starts):
        model = BioheatPINN().to(device)
        model.x_t.data = torch.tensor([random.uniform(-0.5, 0.5)], device=device)
        model.y_t.data = torch.tensor([random.uniform(-0.5, 0.5)], device=device)
        model.z_t.data = torch.tensor([random.uniform(-0.5, 0.5)], device=device)

        tumour_params = [model.x_t, model.y_t, model.z_t,
                         model.r_t, model.Q_max]
        net_params    = list(model.net.parameters())
        optimizer = torch.optim.Adam([
            {'params': net_params,    'lr': 1e-3},
            {'params': tumour_params, 'lr': 1e-2},
        ])
        lambda_pde = None
        history = {'step': [], 'L_data': [], 'L_pde': [], 'loss': []}

        # ── Phase 1: Adam ──
        for step in range(adam_steps):
            model.train(); optimizer.zero_grad()
            T_pred_surf = model(surf_norm)
            L_data = (conf * (T_pred_surf - T_meas)**2).mean()

            ii = int_norm.clone().detach().requires_grad_(True)
            T_i = model(ii)
            lap = compute_laplacian(T_i, ii, extent_meters)
            Q_t = model.Q_tumor(ii)
            residual = (K_TISSUE * lap
                       + OMEGA_B * C_BLOOD * (T_ARTERIAL - T_i)
                       + Q_METAB + Q_t)
            L_pde = (residual**2).mean()

            if lambda_pde is None or step % 1000 == 0:
                lambda_pde = (L_data / (L_pde + 1e-12)).detach()

            loss = L_data + lambda_pde * L_pde
            loss.backward(); optimizer.step()
            model.r_t.data.clamp_(min=2.0)
            model.Q_max.data.clamp_(min=0.0)

            if step % 50 == 0:
                history['step'].append(step)
                history['L_data'].append(L_data.item())
                history['L_pde'].append(L_pde.item())
                history['loss'].append(loss.item())
            if step % 1000 == 0:
                print(f'  [s{start+1}] step {step:4d} | '
                      f'L_data={L_data.item():.4f} '
                      f'L_pde={L_pde.item():.4f} | '
                      f'r={model.r_t.item():.1f}mm '
                      f'Q={model.Q_max.item():.0f}W/m³')

        # ── Phase 2: L-BFGS ──
        lbfgs = torch.optim.LBFGS(model.parameters(), max_iter=20,
                                    line_search_fn='strong_wolfe')
        for blk in range(lbfgs_steps // 20):
            def closure():
                lbfgs.zero_grad()
                Ts = model(surf_norm)
                ld = (conf * (Ts - T_meas)**2).mean()
                ii2 = int_norm.clone().detach().requires_grad_(True)
                Ti  = model(ii2)
                lap2= compute_laplacian(Ti, ii2, extent_meters)
                Qt2 = model.Q_tumor(ii2)
                res2= (K_TISSUE*lap2 + OMEGA_B*C_BLOOD*(T_ARTERIAL-Ti)
                       + Q_METAB + Qt2)
                lp  = (res2**2).mean()
                l   = ld + lambda_pde * lp
                l.backward()
                model.r_t.data.clamp_(min=2.0)
                model.Q_max.data.clamp_(min=0.0)
                return l
            lbfgs.step(closure)

        final_loss = (conf * (model(surf_norm) - T_meas)**2).mean().item()
        if final_loss < best_loss:
            best_loss    = final_loss
            best_state   = {k: v.cpu().clone() for k,v in model.state_dict().items()}
            best_params  = {
                'x_t_norm': model.x_t.item(),
                'y_t_norm': model.y_t.item(),
                'z_t_norm': model.z_t.item(),
                'r_t_mm'  : abs(model.r_t.item()),
                'Q_max'   : model.Q_max.item(),
            }
            best_history = {k: list(v) for k,v in history.items()}
        print(f'  Start {start+1} final data loss: {final_loss:.6f}'
              f'  {"★ best" if final_loss == best_loss else ""}')

    # Denormalise tumour coords to mm
    bp   = best_params
    x_mm = denormalise_coord(bp['x_t_norm'], geo['bbox_min'][0], geo['bbox_max'][0])
    y_mm = denormalise_coord(bp['y_t_norm'], geo['bbox_min'][1], geo['bbox_max'][1])
    z_mm = denormalise_coord(bp['z_t_norm'], geo['bbox_min'][2], geo['bbox_max'][2])

    # ── Convergence plot ──
    if save_plot and best_history:
        pid = geo['patient_id']
        fig_c, ax = plt.subplots(figsize=(10, 4))
        sns.set_theme(style='whitegrid')
        ax.plot(best_history['step'], best_history['L_data'],
                label='Data Loss', color='#1f77b4', lw=2)
        ax.plot(best_history['step'], best_history['L_pde'],
                label='PDE Loss',  color='#ff7f0e', lw=2)
        ax.plot(best_history['step'], best_history['loss'],
                label='Total Loss', color='#2ca02c', ls='--', lw=1.5)
        ax.axvline(adam_steps, color='red', ls=':', lw=1.5,
                   label='L-BFGS transition')
        ax.set_yscale('log'); ax.set_xlabel('Steps'); ax.set_ylabel('Loss')
        ax.set_title(f'PINN Loss — {pid} ({geo["label"]})')
        ax.legend()
        plt.tight_layout()
        plot_path = RESULTS_DIR / f'{pid}_loss_convergence.png'
        plt.savefig(str(plot_path), dpi=150); plt.close(fig_c)
        print(f'  Convergence plot → {plot_path.name}')

    results = {
        'patient_id' : geo['patient_id'],
        'label'      : geo['label'],
        'x_t_mm'     : x_mm,
        'y_t_mm'     : y_mm,
        'z_t_mm'     : z_mm,
        'r_t_mm'     : bp['r_t_mm'],
        'Q_max'      : bp['Q_max'],
        'volume_mm3' : (4/3) * np.pi * bp['r_t_mm']**3,
        'quadrant'   : classify_quadrant(x_mm, y_mm),
        'data_loss'  : best_loss,
    }
    return results, best_state

print('train_pinn_single ready.')

## 11 · FEA forward verification (FEniCSx)

In [ ]:
def stl_to_tet_mesh(stl_path, out_msh_path, mesh_size_mm=3.0):
    import gmsh
    gmsh.initialize()
    gmsh.model.add('breast')
    gmsh.merge(str(stl_path))
    s = gmsh.model.getEntities(2)
    l = gmsh.model.geo.addSurfaceLoop([e[1] for e in s])
    gmsh.model.geo.addVolume([l])
    gmsh.model.geo.synchronize()
    v_ents = gmsh.model.getEntities(3)
    s_ents = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(3, [e[1] for e in v_ents], 1)
    gmsh.model.setPhysicalName(3, 1, 'breast_volume')
    gmsh.model.addPhysicalGroup(2, [e[1] for e in s_ents], 2)
    gmsh.model.setPhysicalName(2, 2, 'breast_surface')
    gmsh.option.setNumber('Mesh.CharacteristicLengthMax', mesh_size_mm)
    gmsh.option.setNumber('Mesh.CharacteristicLengthMin', mesh_size_mm * 0.5)
    gmsh.option.setNumber('Mesh.Algorithm3D', 1)
    gmsh.model.mesh.generate(3)
    gmsh.model.mesh.optimize('Netgen')
    gmsh.write(str(out_msh_path))
    gmsh.finalize()

def run_fea_forward(msh_path, pinn_results):
    from mpi4py import MPI
    import dolfinx, dolfinx.io
    try:
        from dolfinx.io import gmshio
    except ImportError:
        from dolfinx.io import gmsh as gmshio
    from dolfinx import fem
    from dolfinx.fem.petsc import LinearProblem
    import ufl

    fea_import = gmshio.read_from_msh(str(msh_path), MPI.COMM_WORLD, gdim=3)
    msh = fea_import.mesh if hasattr(fea_import, 'mesh') else fea_import[0]
    msh.geometry.x[:] *= 1e-3  # mm → m

    V = fem.functionspace(msh, ('Lagrange', 1))
    T = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)
    x = ufl.SpatialCoordinate(msh)

    x_t  = pinn_results['x_t_mm'] * 1e-3
    y_t  = pinn_results['y_t_mm'] * 1e-3
    z_t  = pinn_results['z_t_mm'] * 1e-3
    r_t  = pinn_results['r_t_mm'] * 1e-3
    Qmax = pinn_results['Q_max']

    d2      = (x[0]-x_t)**2+(x[1]-y_t)**2+(x[2]-z_t)**2
    Q_tumor = Qmax * ufl.exp(-d2 / (r_t**2 + 1e-16))

    k=0.48; wb=0.0005; cb=3600.0; Ta=37.0; Qm=450.0
    h_conv=10.0; T_air=20.0

    z_min = np.min(msh.geometry.x[:, 2])
    def chest_wall(pt): return pt[2] < (z_min + 0.005)
    facets = dolfinx.mesh.locate_entities_boundary(
        msh, msh.topology.dim-1, chest_wall)
    dofs = fem.locate_dofs_topological(V, msh.topology.dim-1, facets)
    bc   = fem.dirichletbc(37.0, dofs, V)

    a = (k*ufl.inner(ufl.grad(T), ufl.grad(v)) +
         wb*cb*T*v)*ufl.dx + h_conv*T*v*ufl.ds
    L = (wb*cb*Ta + Qm + Q_tumor)*v*ufl.dx + h_conv*T_air*v*ufl.ds

    try:
        problem = LinearProblem(a, L, bcs=[bc],
                     petsc_options_prefix='bioheat_',
                     petsc_options={'ksp_type':'cg','pc_type':'gamg'})
    except TypeError:
        problem = LinearProblem(a, L, bcs=[bc],
                     petsc_options={'ksp_type':'cg','pc_type':'gamg'})
    return problem.solve(), msh

def compute_fea_residual(T_fea_sol, msh, surface_pts_mm, T_measured):
    fea_coords = msh.geometry.x * 1e3
    fea_vals   = T_fea_sol.x.array
    tree = cKDTree(fea_coords)
    _, idx = tree.query(surface_pts_mm, k=3)
    T_fea_interp = fea_vals[idx].mean(axis=1)
    residuals    = np.abs(T_fea_interp - T_measured)
    return residuals, residuals.mean(), residuals.max()

print('FEA functions ready.')

## 12 · Single-patient test run

In [ ]:
# ── Run the full pipeline for one patient before batching all 122 ──
TEST_IDX = 0
item     = dataset[TEST_IDX]
pid      = item['patient_id']
print(f'Test patient: {pid} ({item["label"]})')

patient_dir = RESULTS_DIR / pid
patient_dir.mkdir(exist_ok=True)

# Step 1: NeRF extraction
print('\n[1/4] NeRF volume extraction…')
tiffs_n  = item['tiffs_norm'].to(device)
masks_   = item['masks'].to(device)
tmin_ref = item['tmin'][2].item()   # frontal-view anchor
tmax_ref = item['tmax'][2].item()
sigma_grid, T_grid = extract_3d_volume(encoder, mlp, tiffs_n, masks_, CFG, device)

# Step 2: Geometry preprocessing
print('\n[2/4] Geometry preprocessing…')
geo = process_patient_geometry_from_nerf(sigma_grid, T_grid,
                                          tmin_ref, tmax_ref, CFG)
geo['patient_id'] = pid
geo['label']      = item['label']

stl_path = patient_dir / f'{pid}.stl'
save_stl_binary(str(stl_path), geo['surface_pts'], geo['faces'])
np.save(str(patient_dir / f'{pid}_T_measured.npy'), geo['T_measured'])
np.save(str(patient_dir / f'{pid}_surf_pts.npy'),   geo['surface_pts'])

# Step 3: PINN
print('\n[3/4] PINN training…')
pinn_results, pinn_state = train_pinn_single(
    geo, device, n_starts=3, adam_steps=3000, lbfgs_steps=100
)
torch.save(pinn_state, str(patient_dir / f'{pid}_pinn.pth'))

# Step 4: FEA
print('\n[4/4] FEA forward verification…')
msh_path = patient_dir / f'{pid}.msh'
try:
    stl_to_tet_mesh(str(stl_path), str(msh_path), mesh_size_mm=3.0)
    T_fea_sol, fea_msh = run_fea_forward(str(msh_path), pinn_results)
    residuals, mean_res, max_res = compute_fea_residual(
        T_fea_sol, fea_msh, geo['surface_pts'], geo['T_measured'])
    pinn_results['fea_mean_residual'] = mean_res
    pinn_results['fea_max_residual']  = max_res
    pinn_results['fea_status'] = 'OK' if mean_res < 1.5 else 'HIGH_RESIDUAL'
    print(f'FEA residual: mean={mean_res:.3f}°C  max={max_res:.3f}°C  '
          f'→ {pinn_results["fea_status"]}')
except Exception as e:
    print(f'FEA failed: {e}')
    pinn_results['fea_mean_residual'] = np.nan
    pinn_results['fea_status'] = f'FAILED: {e}'

print(f'\n✓ {pid} | Q_max={pinn_results["Q_max"]:.0f} W/m³ | '
      f'r={pinn_results["r_t_mm"]:.1f}mm | '
      f'quadrant={pinn_results["quadrant"]}')

## 13 · Tumour location visualisation

In [ ]:
def plot_tumor_location(geo, pinn_results, save_html=True):
    verts     = geo['surface_pts']
    faces     = geo['faces']
    temp_vals = geo['T_measured']
    pid       = geo['patient_id']
    x_t = pinn_results['x_t_mm']
    y_t = pinn_results['y_t_mm']
    z_t = pinn_results['z_t_mm']
    r_t = pinn_results['r_t_mm']

    # Transparent geometry mesh
    mesh_trace = go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=temp_vals, colorscale='Inferno',
        cmin=temp_vals.min(), cmax=temp_vals.max(),
        opacity=0.25, name='Surface (thermal)', showscale=True,
        colorbar=dict(title='°C', x=1.02),
    )

    # Tumour sphere (parametric)
    u = np.linspace(0, 2*np.pi, 40)
    v = np.linspace(0, np.pi, 40)
    xs = x_t + r_t * np.outer(np.cos(u), np.sin(v))
    ys = y_t + r_t * np.outer(np.sin(u), np.sin(v))
    zs = z_t + r_t * np.outer(np.ones(40), np.cos(v))
    tumor_trace = go.Surface(
        x=xs, y=ys, z=zs,
        colorscale=[[0,'#FF4500'],[1,'#FF4500']],
        opacity=0.7, showscale=False,
        name=f'Tumour (r={r_t:.1f}mm)',
    )

    # Centroid marker
    center_trace = go.Scatter3d(
        x=[x_t], y=[y_t], z=[z_t],
        mode='markers+text',
        marker=dict(size=8, color='red', symbol='cross'),
        text=[f'Q={pinn_results["Q_max"]:.0f}W/m³'],
        textposition='top center',
        name='Heat source centroid',
    )

    fig = go.Figure(data=[mesh_trace, tumor_trace, center_trace])
    fig.update_layout(
        title=dict(
            text=(f'PINN Tumour Localisation — {pid}<br>'
                  f'<sup>({pinn_results["quadrant"]}) | '
                  f'r={r_t:.1f}mm | '
                  f'Q={pinn_results["Q_max"]:.0f}W/m³ | '
                  f'FEA residual={pinn_results.get("fea_mean_residual",float("nan")):.2f}°C</sup>'),
            font=dict(size=14),
        ),
        scene=dict(
            aspectmode='data',
            xaxis=dict(title='X (RL↔LL) mm'),
            yaxis=dict(title='Y (Inf↑Sup) mm'),
            zaxis=dict(title='Z (Depth) mm'),
            camera=dict(eye=dict(x=1.5, y=0.6, z=1.0)),
        ),
        margin=dict(l=0,r=0,b=0,t=70),
        width=950, height=750,
    )
    if save_html:
        html_path = RESULTS_DIR / f"{pid}_tumor_localisation.html"
        fig.write_html(str(html_path), include_plotlyjs='cdn')
        print(f'Saved → {html_path}')
    return fig

fig2 = plot_tumor_location(geo, pinn_results)
fig2.show()

## 14 · Full 122-patient cohort loop

In [ ]:
all_results = []

for patient_idx in tqdm(range(len(dataset)), desc='Cohort pipeline'):
    item = dataset[patient_idx]
    pid  = item['patient_id']
    lab  = item['label']
    print(f'\n{"═"*55}')
    print(f'Patient {patient_idx+1}/{len(dataset)}: {pid} ({lab})')
    print(f'{"═"*55}')

    try:
        patient_dir = RESULTS_DIR / pid
        patient_dir.mkdir(exist_ok=True, parents=True)

        # ── 1. NeRF extraction ──
        tiffs_n  = item['tiffs_norm'].to(device)
        masks_   = item['masks'].to(device)
        tmin_ref = item['tmin'][2].item()   # frontal view anchor
        tmax_ref = item['tmax'][2].item()
        sigma_grid, T_grid = extract_3d_volume(
            encoder, mlp, tiffs_n, masks_, CFG, device)

        # ── 2. Geometry preprocessing ──
        geo = process_patient_geometry_from_nerf(
            sigma_grid, T_grid, tmin_ref, tmax_ref, CFG)
        geo['patient_id'] = pid
        geo['label']      = lab

        stl_path = patient_dir / f'{pid}.stl'
        save_stl_binary(str(stl_path), geo['surface_pts'], geo['faces'])
        np.save(str(patient_dir/f'{pid}_T_measured.npy'), geo['T_measured'])
        np.save(str(patient_dir/f'{pid}_surf_pts.npy'),   geo['surface_pts'])

        # ── 3. PINN ──
        pinn_results, pinn_state = train_pinn_single(
            geo, device, n_starts=3, adam_steps=3000, lbfgs_steps=100)
        torch.save(pinn_state, str(patient_dir/f'{pid}_pinn.pth'))

        # ── 4. FEA ──
        msh_path = patient_dir / f'{pid}.msh'
        try:
            stl_to_tet_mesh(str(stl_path), str(msh_path), mesh_size_mm=3.0)
            T_sol, fea_msh = run_fea_forward(str(msh_path), pinn_results)
            res, mean_res, max_res = compute_fea_residual(
                T_sol, fea_msh, geo['surface_pts'], geo['T_measured'])
            np.save(str(patient_dir/f'{pid}_fea_residuals.npy'), res)
            pinn_results['fea_mean_residual'] = mean_res
            pinn_results['fea_max_residual']  = max_res
            pinn_results['fea_status'] = \
                'OK' if mean_res < 1.5 else 'HIGH_RESIDUAL'
            print(f'  FEA: mean={mean_res:.3f}°C '
                  f'max={max_res:.3f}°C '
                  f'→ {pinn_results["fea_status"]}')
        except Exception as e:
            print(f'  ⚠ FEA failed: {e}')
            pinn_results['fea_mean_residual'] = np.nan
            pinn_results['fea_max_residual']  = np.nan
            pinn_results['fea_status']        = f'FAILED: {e}'

        all_results.append(pinn_results)
        print(f'  ✓ Q={pinn_results["Q_max"]:.0f}W/m³ | '
              f'r={pinn_results["r_t_mm"]:.1f}mm | '
              f'{pinn_results["quadrant"]}')

    except Exception as e:
        import traceback
        print(f'  ✗ PIPELINE FAILED: {e}')
        traceback.print_exc()
        all_results.append({
            'patient_id': pid, 'label': lab,
            'Q_max': np.nan, 'r_t_mm': np.nan,
            'fea_status': f'PIPELINE_FAILED: {e}',
        })

# ── Save master CSV ──
df       = pd.DataFrame(all_results)
csv_path = RESULTS_DIR / 'pinn_fea_results.csv'
df.to_csv(str(csv_path), index=False)
print(f'\nSaved → {csv_path}')
print(f'Processed: {len(df)} | Failed: {df["Q_max"].isna().sum()}')
print(df[['patient_id','label','Q_max','r_t_mm',
          'quadrant','fea_mean_residual']].to_string())

## 15 · Cohort summary visualisation

In [ ]:
df = pd.read_csv(str(RESULTS_DIR / 'pinn_fea_results.csv'))
df_ok = df[df['Q_max'].notna()]

fig_s, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.set_theme(style='whitegrid')

# Q_max distribution by label
sns.boxplot(data=df_ok, x='label', y='Q_max', ax=axes[0],
            palette={'benign':'#5B9BD5','malignant':'#ED7D31'})
axes[0].set_title('Q_max (W/m³) by Label')
axes[0].set_ylabel('Q_max (W/m³)')

# Tumour radius distribution
sns.boxplot(data=df_ok, x='label', y='r_t_mm', ax=axes[1],
            palette={'benign':'#5B9BD5','malignant':'#ED7D31'})
axes[1].set_title('Estimated Tumour Radius (mm)')
axes[1].set_ylabel('r_t (mm)')

# FEA mean residual
df_fea = df_ok[df_ok['fea_status'] == 'OK']
sns.histplot(data=df_fea, x='fea_mean_residual', bins=15,
             ax=axes[2], color='#70AD47')
axes[2].axvline(1.5, color='red', ls='--', label='1.5°C threshold')
axes[2].set_title('FEA Mean Residual (°C)')
axes[2].set_xlabel('Mean |T_FEA - T_measured| (°C)')
axes[2].legend()

plt.suptitle('TherMAM-NeRF + PINN Cohort Results (n=122)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
summary_path = RESULTS_DIR / 'cohort_pinn_summary.png'
plt.savefig(str(summary_path), dpi=150)
plt.show()
print(f'Saved → {summary_path}')

# Quadrant distribution
print('\nQuadrant distribution:')
print(df_ok['quadrant'].value_counts().to_string())
print(f'\nFEA pass rate (mean residual < 1.5°C): '
      f'{len(df_fea)}/{len(df_ok)} '
      f'({100*len(df_fea)/len(df_ok):.1f}%)')